# Лабораторная 1: GRU seq2seq с Luong attention для reverse-sequence

## Цель
Реализовать `GRU` encoder-decoder с `Luong attention`, который разворачивает входную последовательность в обратном порядке.

Эта лабораторная продолжает `seq2seq` из блока `RNN`, но убирает fixed-context bottleneck: decoder теперь может смотреть на все позиции `encoder_output`, а не только на один финальный вектор состояния.


## Контракт данных
Используются целочисленные токены:
- `1..9` — содержательные значения;
- `PAD=0` — заполнение;
- `SOS=10` — старт декодера;
- `EOS=11` — конец последовательности.

Используется та же reverse-задача, что и в ЛР3 по `RNN`, но теперь:
- `ENC_LEN = 10`;
- реальная длина последовательности случайна в диапазоне `4..10`.

Формируются три связанных тензора:
- `encoder_input`;
- `decoder_input`;
- `decoder_target`.


## Таблица форм тензоров

| Тензор | Форма | Смысл | Где используется |
|---|---|---|---|
| `encoder_input` | `(N, T_in)` | Вход encoder | `model.fit` / `model.predict` |
| `decoder_input` | `(N, T_out)` | Вход decoder со сдвигом | `model.fit` / `model.predict` |
| `decoder_target` | `(N, T_out, 1)` | Истинные выходные токены | Функция потерь |
| `encoder_outputs` | `(N, T_in, H)` | Состояния encoder по всем входным позициям | Attention |
| `decoder_outputs` | `(N, T_out, H)` | Состояния decoder по всем выходным шагам | Attention / head |
| `context` | `(N, T_out, H)` | Контекст из encoder для каждого decoder-шага | Attention |
| `attention_scores` | `(N, T_out, T_in)` | Веса внимания по входным позициям | Диагностика |
| `probs` | `(N_test, T_out, V)` | Распределения вероятностей по словарю | `model.predict` |
| `preds` | `(N_test, T_out)` | Предсказанные индексы | Оценка |
| `exact_match` | скаляр | Доля полностью верных последовательностей | Итоговая метрика |


## Контракт модели
- В `model.fit` передается список входов `[encoder_input, decoder_input]` и `decoder_target`.
- `encoder` должен вернуть и всю последовательность состояний, и финальное состояние.
- `decoder` инициализируется `encoder_state`, но на каждом шаге дополнительно смотрит на `encoder_outputs` через `Attention(score_mode="dot")`.
- Выходной слой получает конкатенацию `[decoder_outputs; context]`.
- Для функции потерь `sparse_categorical_crossentropy` целевые значения задаются целыми индексами классов.
- Для диагностики строится отдельный `analysis_model`, который возвращает промежуточные тензоры `encoder_outputs`, `decoder_outputs`, `context`, `attention_scores`.


## Мини-теория
В этой лабораторной используется `cross-attention` в варианте Luong:

$$
q_t = h_t^{dec}, \quad k_s = h_s^{enc}, \quad v_s = h_s^{enc}
$$

$$
e_{t,s} = q_t^\top k_s, \quad
\alpha_{t,s} = \mathrm{softmax}_s(e_{t,s}), \quad
c_t = \sum_{s=1}^{T_{in}} \alpha_{t,s} v_s
$$

$$
z_t = [h_t^{dec}; c_t], \quad s_t = W_{out} z_t + b_{out}, \quad
\hat{y}_t = \mathrm{softmax}(s_t)
$$

Здесь:
- `query` идет от текущего шага decoder;
- `key/value` идут от всех шагов encoder;
- `attention_scores[t, s]` показывает, на какую входную позицию модель смотрит на шаге `t`.


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed(42)
print('Версия TensorFlow:', tf.__version__)


## Генерация данных
**Что сделать:** подготовить `encoder_input`, `decoder_input`, `decoder_target` со сдвигом на один шаг.

**Почему:** decoder по-прежнему обучается по схеме `teacher forcing`, но теперь задача специально сделана длиннее, чтобы plain `seq2seq` чаще упирался в одно фиксированное состояние.

**Ожидаемые формы:** `(N, T_in)`, `(N, T_out)`, `(N, T_out, 1)`.

**Частая ошибка:** неверный сдвиг между `decoder_input` и `decoder_target`.


In [ ]:
PAD_ID = 0
SOS_ID = 10
EOS_ID = 11
VOCAB_SIZE = 12
ENC_LEN = 10
DEC_LEN = ENC_LEN + 1


def decode_token(tok: int) -> str:
    mapping = {PAD_ID: 'PAD', SOS_ID: 'SOS', EOS_ID: 'EOS'}
    return mapping.get(int(tok), str(int(tok)))


def make_one_sample(min_len: int = 4, max_len: int = ENC_LEN):
    length = np.random.randint(min_len, max_len + 1)
    seq = np.random.randint(1, 10, size=length, dtype=np.int32)
    # TODO 1: получите обратную последовательность
    rev = ...
    return seq, rev


def pad_sequence(seq, target_len, pad_value=PAD_ID):
    out = np.full((target_len,), pad_value, dtype=np.int32)
    out[:len(seq)] = seq
    return out


def make_dataset(n_samples: int = 8000):
    encoder_input = np.zeros((n_samples, ENC_LEN), dtype=np.int32)
    decoder_input = np.zeros((n_samples, DEC_LEN), dtype=np.int32)
    decoder_target = np.zeros((n_samples, DEC_LEN), dtype=np.int32)

    for i in range(n_samples):
        seq, rev = make_one_sample()
        enc = pad_sequence(seq, ENC_LEN)

        # TODO 2: сформируйте decoder_input: [SOS] + rev
        dec_in = ...
        # TODO 3: сформируйте decoder_target: rev + [EOS]
        dec_out = ...

        encoder_input[i] = pad_sequence(enc, ENC_LEN)
        decoder_input[i] = pad_sequence(dec_in, DEC_LEN)
        decoder_target[i] = pad_sequence(dec_out, DEC_LEN)

    return encoder_input, decoder_input, decoder_target[..., None]


enc_in, dec_in, dec_tgt = make_dataset()

enc_train, enc_test, dec_in_train, dec_in_test, dec_tgt_train, dec_tgt_test = train_test_split(
    enc_in,
    dec_in,
    dec_tgt,
    test_size=0.2,
    random_state=42,
)

print('Форма enc_train    :', enc_train.shape)
print('Форма dec_in_train :', dec_in_train.shape)
print('Форма dec_tgt_train:', dec_tgt_train.shape)
print('Пример encoder_input :', enc_train[0])
print('Пример decoder_input :', dec_in_train[0])
print('Пример decoder_target:', dec_tgt_train[0, :, 0])


In [ ]:
# Мини-проверка данных
assert enc_in.shape[1] == ENC_LEN and enc_in.ndim == 2
assert dec_in.shape[1] == DEC_LEN and dec_in.ndim == 2
assert dec_tgt.shape == (enc_in.shape[0], DEC_LEN, 1)
print('Мини-проверка данных: OK')


## Модель
Используется полный `Functional API`, потому что attention требует несколько промежуточных тензоров.

**Что сделать:** собрать модель из блоков:
- `Embedding(mask_zero=True)` для encoder;
- `GRU(return_sequences=True, return_state=True)` для encoder;
- `Embedding(mask_zero=True)` для decoder;
- `GRU(return_sequences=True, return_state=True, initial_state=encoder_state)` для decoder;
- `Attention(score_mode="dot")`;
- `Concatenate([decoder_outputs, context])`;
- `Dense(VOCAB_SIZE, activation="softmax")`.


In [ ]:
def build_model(vocab_size: int = VOCAB_SIZE, emb_dim: int = 32, latent_dim: int = 64):
    encoder_inputs = tf.keras.layers.Input(shape=(ENC_LEN,), name='encoder_inputs')
    # TODO 4: создайте encoder embedding с mask_zero=True
    enc_emb = ...

    # TODO 5: создайте encoder GRU c return_sequences=True и return_state=True
    enc_gru = ...
    # TODO 6: получите encoder_outputs и encoder_state
    encoder_outputs, encoder_state = ...

    decoder_inputs = tf.keras.layers.Input(shape=(DEC_LEN,), name='decoder_inputs')
    # TODO 7: создайте decoder embedding с mask_zero=True
    dec_emb = ...

    # TODO 8: создайте decoder GRU c return_sequences=True и return_state=True
    dec_gru = ...
    # TODO 9: получите decoder_outputs с initial_state=encoder_state
    decoder_outputs, _ = ...

    attn = tf.keras.layers.Attention(score_mode='dot', name='cross_attention')
    # TODO 10: получите context и attention_scores
    context, attention_scores = ...

    # TODO 11: объедините decoder_outputs и context по последней оси
    merged = ...
    # TODO 12: добавьте Dense(vocab_size, activation='softmax')
    outputs = ...

    model = tf.keras.Model([encoder_inputs, decoder_inputs], outputs, name='gru_seq2seq_attention')
    analysis_model = tf.keras.Model(
        [encoder_inputs, decoder_inputs],
        [encoder_outputs, decoder_outputs, context, attention_scores, outputs],
        name='attention_analysis_model',
    )

    # TODO 13: скомпилируйте модель (adam, sparse_categorical_crossentropy, accuracy)
    model.compile(...)
    return model, analysis_model


model, analysis_model = build_model()
model.summary()


In [ ]:
# Мини-проверка модели
sample_enc = enc_train[:2]
sample_dec = dec_in_train[:2]

encoder_outputs_s, decoder_outputs_s, context_s, attention_scores_s, probs_s = analysis_model.predict(
    [sample_enc, sample_dec],
    verbose=0,
)

assert model.output_shape == (None, DEC_LEN, VOCAB_SIZE)
assert encoder_outputs_s.shape[:2] == (2, ENC_LEN)
assert decoder_outputs_s.shape[:2] == (2, DEC_LEN)
assert context_s.shape == decoder_outputs_s.shape
assert attention_scores_s.shape == (2, DEC_LEN, ENC_LEN)
assert probs_s.shape == (2, DEC_LEN, VOCAB_SIZE)

row_sums = attention_scores_s.sum(axis=-1)
valid_decoder_positions = (sample_dec != PAD_ID)
assert np.allclose(row_sums[valid_decoder_positions], 1.0, atol=1e-5)
print('Мини-проверка модели: OK')


## Трассировка одного примера через модель
Проверка форм тензоров:
1. входы encoder и decoder;
2. выход encoder по всем входным позициям;
3. выход decoder по всем шагам;
4. контекст attention;
5. итоговое распределение вероятностей.


In [ ]:
sample_enc = enc_train[:1]
sample_dec = dec_in_train[:1]

encoder_outputs_s, decoder_outputs_s, context_s, attention_scores_s, probs_s = analysis_model.predict(
    [sample_enc, sample_dec],
    verbose=0,
)

print('Вход encoder_input      :', sample_enc.shape)
print('Вход decoder_input      :', sample_dec.shape)
print('После encoder GRU       :', encoder_outputs_s.shape)
print('После decoder GRU       :', decoder_outputs_s.shape)
print('После attention context :', context_s.shape)
print('attention_scores        :', attention_scores_s.shape)
print('После выходного слоя    :', probs_s.shape)
print('Суммы attention по encoder-оси:', np.round(attention_scores_s[0, :5].sum(axis=-1), 3))
print('Первые 3 предсказанных индекса:', probs_s.argmax(axis=-1)[0, :3])


## Как идет обучение внутри эпохи
Для каждого мини-батча выполняются шаги:
1. encoder читает всю входную последовательность и выдает `encoder_outputs` и финальное состояние;
2. decoder получает `decoder_input` и стартует из `encoder_state`;
3. на каждом шаге decoder attention пересчитывает веса по всем позициям encoder;
4. выходной слой получает `[decoder_outputs; context]` и строит вероятности токенов;
5. потери суммируются по всем шагам, затем выполняется `BPTT`.

Ключевое отличие от plain `seq2seq`: decoder не обязан помнить весь вход только через один вектор, потому что может заново читать encoder-память на каждом шаге.


## Обучение
В `fit` подается пара входов `[encoder_input, decoder_input]` и целевой тензор `decoder_target`.

Для контроля обобщения используется `validation_split=0.2` на обучающей части.


In [ ]:
def train_model(model, enc_train, dec_in_train, dec_tgt_train, epochs: int = 18):
    # TODO 14: обучите модель через model.fit
    history = ...
    return history


history = train_model(model, enc_train, dec_in_train, dec_tgt_train)


In [ ]:
# Мини-проверка обучения
assert len(history.history['loss']) > 0
assert 'val_loss' in history.history
assert np.isfinite(history.history['loss']).all()
print('Финальный val_token_accuracy:', round(history.history['val_accuracy'][-1], 4))
print('Мини-проверка обучения: OK')


## Оценка и диагностика
Считаются две основные метрики:
- `token_accuracy` — доля корректных токенов;
- `exact_match` — доля последовательностей, где все значимые токены предсказаны верно.

Дополнительно внимание проверяется качественно через heatmap одного примера.


In [ ]:
def evaluate_model(model, enc_test, dec_in_test, dec_tgt_test):
    loss, token_acc = model.evaluate([enc_test, dec_in_test], dec_tgt_test, verbose=0)
    probs = model.predict([enc_test, dec_in_test], verbose=0)
    preds = probs.argmax(axis=-1)
    target = dec_tgt_test[:, :, 0]
    mask = (target != PAD_ID)

    # TODO 15: вычислите exact_match только по значимым позициям
    exact_match = ...

    return {
        'loss': float(loss),
        'token_accuracy': float(token_acc),
        'exact_match': float(exact_match),
        'preds': preds,
        'target': target,
        'mask': mask,
        'probs': probs,
    }


metrics = evaluate_model(model, enc_test, dec_in_test, dec_tgt_test)
print({k: v for k, v in metrics.items() if k in ('loss', 'token_accuracy', 'exact_match')})


In [ ]:
# Мини-проверка метрик
assert 0.0 <= metrics['token_accuracy'] <= 1.0
assert 0.0 <= metrics['exact_match'] <= 1.0
if metrics['exact_match'] >= 0.85:
    print('Целевой порог достигнут: exact_match >= 0.85')
else:
    print('Целевой порог не достигнут: проверьте attention, teacher forcing и число эпох')


## Карта внимания одного примера
Ниже строится heatmap для одного тестового примера после обучения.

Интерпретация:
- столбцы — входные позиции encoder;
- строки — шаги decoder по значимым целевым токенам;
- яркие клетки показывают, на какие входные позиции модель опирается сильнее всего.

Для reverse-задачи обычно наблюдается движение фокуса примерно по антидиагонали.


In [ ]:
sample_enc = enc_test[:1]
sample_dec = dec_in_test[:1]
sample_target = dec_tgt_test[:1, :, 0]
sample_probs = model.predict([sample_enc, sample_dec], verbose=0)
sample_preds = sample_probs.argmax(axis=-1)
_, _, _, sample_attention, _ = analysis_model.predict([sample_enc, sample_dec], verbose=0)

enc_valid_len = int(np.count_nonzero(sample_enc[0] != PAD_ID))
tgt_valid_len = int(np.count_nonzero(sample_target[0] != PAD_ID))

score_map = sample_attention[0, :tgt_valid_len, :enc_valid_len]
enc_labels = [decode_token(tok) for tok in sample_enc[0, :enc_valid_len]]
tgt_labels = [decode_token(tok) for tok in sample_target[0, :tgt_valid_len]]
pred_labels = [decode_token(tok) for tok in sample_preds[0, :tgt_valid_len]]
row_labels = [f'{t}/{p}' for t, p in zip(tgt_labels, pred_labels)]

plt.figure(figsize=(max(6, enc_valid_len), max(4, int(0.8 * tgt_valid_len) + 2)))
plt.imshow(score_map, aspect='auto', cmap='viridis')
plt.colorbar(label='attention weight')
plt.xticks(range(enc_valid_len), enc_labels)
plt.yticks(range(tgt_valid_len), row_labels)
plt.xlabel('Позиции encoder_input')
plt.ylabel('target / pred')
plt.title('Карта внимания одного примера')

for i in range(score_map.shape[0]):
    for j in range(score_map.shape[1]):
        value = score_map[i, j]
        color = 'white' if value > 0.5 else 'black'
        plt.text(j, i, f'{value:.2f}', ha='center', va='center', color=color, fontsize=8)

plt.tight_layout()
plt.show()

print('encoder_input :', enc_labels)
print('decoder_target:', tgt_labels)
print('preds         :', pred_labels)


## Что ожидать на практике
- `token_accuracy` обычно растет быстрее `exact_match`.
- По сравнению с plain `GRU seq2seq` модель с attention обычно быстрее исправляет ошибки на длинных последовательностях.
- Для корректной реализации heatmap на reverse-задаче часто показывает движение фокуса примерно по антидиагонали.
- Для корректной реализации ожидается достижение `exact_match >= 0.85`.


In [ ]:
# Визуализация динамики обучения
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Динамика функции потерь')
plt.xlabel('Эпоха')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_token_accuracy')
plt.plot(history.history['val_accuracy'], label='val_token_accuracy')
plt.title('Динамика token_accuracy')
plt.xlabel('Эпоха')
plt.legend()

plt.tight_layout()
plt.show()


## Бонус: сравнение с plain GRU seq2seq
Эта секция необязательна.

Ниже определен baseline без attention на том же датасете. По умолчанию он не обучается, чтобы не удлинять основной проход по ноутбуку.

Если хотите сравнить fixed-context bottleneck с attention на практике, установите `RUN_BASELINE = True`.


In [ ]:
def build_plain_seq2seq_model(vocab_size: int = VOCAB_SIZE, emb_dim: int = 32, latent_dim: int = 64):
    encoder_inputs = tf.keras.layers.Input(shape=(ENC_LEN,), name='plain_encoder_inputs')
    enc_emb = tf.keras.layers.Embedding(vocab_size, emb_dim, mask_zero=True, name='plain_enc_embedding')(encoder_inputs)
    encoder_state = tf.keras.layers.GRU(latent_dim, return_state=True, name='plain_enc_gru')(enc_emb)[1]

    decoder_inputs = tf.keras.layers.Input(shape=(DEC_LEN,), name='plain_decoder_inputs')
    dec_emb = tf.keras.layers.Embedding(vocab_size, emb_dim, mask_zero=True, name='plain_dec_embedding')(decoder_inputs)
    decoder_outputs = tf.keras.layers.GRU(
        latent_dim,
        return_sequences=True,
        return_state=True,
        name='plain_dec_gru',
    )(dec_emb, initial_state=encoder_state)[0]

    outputs = tf.keras.layers.Dense(vocab_size, activation='softmax', name='plain_output_head')(decoder_outputs)
    model = tf.keras.Model([encoder_inputs, decoder_inputs], outputs, name='plain_gru_seq2seq')
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


RUN_BASELINE = False

if RUN_BASELINE:
    baseline_model = build_plain_seq2seq_model()
    _ = train_model(baseline_model, enc_train, dec_in_train, dec_tgt_train, epochs=12)
    baseline_metrics = evaluate_model(baseline_model, enc_test, dec_in_test, dec_tgt_test)
    print({
        'attention_token_accuracy': round(metrics['token_accuracy'], 4),
        'attention_exact_match': round(metrics['exact_match'], 4),
        'plain_token_accuracy': round(baseline_metrics['token_accuracy'], 4),
        'plain_exact_match': round(baseline_metrics['exact_match'], 4),
    })
else:
    print('Секция пропущена. Установите RUN_BASELINE = True, чтобы сравнить с plain GRU seq2seq без attention.')


## Вопросы для самопроверки
1. Почему одного `encoder_state` может не хватать для длинной последовательности?
2. Что в этой лабораторной играет роли `query`, `key`, `value`?
3. Чем `cross-attention` отличается от `self-attention`?
4. Почему `Transformer` логично изучать уже после этой лабораторной, а не вместо нее?
5. Почему `token_accuracy` может быть высокой, а `exact_match` заметно ниже?


## Типичные ошибки (симптом -> причина -> исправление)
- `attention_scores` имеет неправильную форму -> перепутаны `query` и `value` -> вызывать `Attention([decoder_outputs, encoder_outputs])`.
- Attention почти бесполезен -> encoder возвращает только один вектор -> включить `return_sequences=True` у `enc_gru`.
- `exact_match` низкий при нормальном `token_accuracy` -> модель ошибается хотя бы в одном шаге многих последовательностей -> проверить `teacher forcing`, конкатенацию `[decoder_outputs; context]` и число эпох.
- Heatmap плохо читается -> PAD-позиции попали в визуализацию -> показывать только значимые строки и столбцы.
